# RAASTA — YOLOv8 training (Module 2 + Module 5)

One model for: `pothole`, `crack`, `speed_bump`, `person`, `cow`, `dog`, `goat`, `horse`

## Before you run
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Free [Roboflow](https://roboflow.com) account → **Account → Roboflow API** → copy key
3. Paste key in the next cell → **Runtime → Run all**

When done: download `raasta_m2_m5.tflite` → put in `assets/models/` on your PC.

In [ ]:
#@title 1) Paste Roboflow API key + install
ROBOFLOW_API_KEY = ""  #@param {type:"string"}

assert ROBOFLOW_API_KEY.strip(), "Paste your Roboflow API key first"

# Fix Colab Pillow mismatch (ImportError: _Ink from PIL._typing), then install stack.
!pip -q uninstall -y pillow pillow-simd 2>/dev/null
!pip -q install -U "pillow==11.2.1" "ultralytics>=8.3.0" roboflow opencv-python-headless pyyaml

import os, shutil, random, yaml
from pathlib import Path
from collections import Counter

import torch
from ultralytics import YOLO  # fail early if PIL still broken

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("Ultralytics import: OK")
assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU, then re-run from top"

In [ ]:
#@title 2) Class map
CLASS_NAMES = [
    "pothole", "crack", "speed_bump",
    "person", "cow", "dog", "goat", "horse",
]
CLASS_TO_ID = {n: i for i, n in enumerate(CLASS_NAMES)}

LABEL_ALIASES = {
    "pothole": "pothole", "potholes": "pothole", "hole": "pothole",
    "crack": "crack", "cracks": "crack", "alligator": "crack",
    "alligator crack": "crack", "longitudinal crack": "crack", "transverse crack": "crack",
    "speed_bump": "speed_bump", "speed-bump": "speed_bump", "speedbump": "speed_bump",
    "speed bump": "speed_bump", "speed breaker": "speed_bump", "speed-breaker": "speed_bump",
    "bump": "speed_bump", "hump": "speed_bump", "speed hump": "speed_bump",
    "unmarked bump": "speed_bump",
    "person": "person", "pedestrian": "person", "people": "person",
    "cow": "cow", "cattle": "cow", "ox": "cow",
    "dog": "dog",
    "goat": "goat", "sheep": "goat",
    "horse": "horse",
}

def canonical(raw: str):
    key = raw.strip()
    if key in LABEL_ALIASES:
        return LABEL_ALIASES[key]
    low = key.lower().replace("_", " ").replace("-", " ")
    for a, c in LABEL_ALIASES.items():
        if a.lower().replace("_", " ").replace("-", " ") == low:
            return c
    return None

WORK = Path("/content/raasta")
RAW = WORK / "raw"
MERGED = WORK / "merged"
shutil.rmtree(WORK, ignore_errors=True)
RAW.mkdir(parents=True)
print("OK")

In [ ]:
#@title 3) Download Roboflow datasets (M2 road + M5 objects)
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY.strip())
sources = []

def download_rf(workspace, project, version, folder_name):
    out = RAW / folder_name
    if out.exists() and any(out.rglob("*.jpg")):
        print("[cached]", folder_name)
        return out
    print("[download]", workspace, project, version)
    ds = rf.workspace(workspace).project(project).version(version).download(
        "yolov8", location=str(out)
    )
    return Path(ds.location)

# M2 — road damage / speed bumps (your priority list)
rf_sets = [
    ("speed-bump-detection", "speed-bump-detection-se0eh", 15, "speed_bump_v15"),
    ("nsip-project", "road-degradation-beta", 1, "road_degradation"),
    ("pothole-detection-1nczj", "speed-unmarked-bumb", 1, "unmarked_bump"),
]

# M5 — people / animals (best-effort; skips are OK)
rf_sets += [
    ("joseph-nelson", "people", 1, "people_rf"),
    ("brad-dwyer", "aerial-cows", 1, "cows_rf"),
]

for ws, proj, ver, name in rf_sets:
    try:
        sources.append(download_rf(ws, proj, ver, name))
    except Exception as e:
        print("[warn] skip", name, "→", e)

print("Downloaded:", [p.name for p in sources])
assert sources, "No datasets downloaded — check API key / internet"

In [ ]:
#@title 4) Optional Open Images boost (person/cow/dog/goat/horse)
# Safe to skip if this fails — pretrained YOLO backbone still helps M5.
try:
    %pip -q install fiftyone
    import fiftyone.zoo as foz

    oi = foz.load_zoo_dataset(
        "open-images-v7",
        split="validation",
        label_types=["detections"],
        classes=["Person", "Cattle", "Dog", "Goat", "Horse"],
        max_samples=2500,
        dataset_name="raasta-oi",
    )
    oi_map = {
        "Person": "person", "Cattle": "cow", "Dog": "dog",
        "Goat": "goat", "Horse": "horse",
    }
    oi_yolo = RAW / "openimages_yolo"
    for split in ("train", "val"):
        (oi_yolo / "images" / split).mkdir(parents=True, exist_ok=True)
        (oi_yolo / "labels" / split).mkdir(parents=True, exist_ok=True)

    n = 0
    samples = list(oi)
    random.shuffle(samples)
    for i, sample in enumerate(samples):
        if sample.ground_truth is None:
            continue
        lines = []
        for det in sample.ground_truth.detections:
            cname = oi_map.get(det.label)
            if not cname:
                continue
            x, y, bw, bh = det.bounding_box
            cx, cy = x + bw / 2, y + bh / 2
            lines.append(f"{CLASS_TO_ID[cname]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
        if not lines:
            continue
        split = "val" if i % 6 == 0 else "train"
        stem = f"oi_{i:06d}"
        ext = Path(sample.filepath).suffix.lower()
        shutil.copy2(sample.filepath, oi_yolo / "images" / split / f"{stem}{ext}")
        (oi_yolo / "labels" / split / f"{stem}.txt").write_text("\n".join(lines) + "\n")
        n += 1

    (oi_yolo / "data.yaml").write_text(yaml.safe_dump({
        "names": {i: n for i, n in enumerate(CLASS_NAMES)},
        "path": str(oi_yolo),
        "train": "images/train",
        "val": "images/val",
    }))
    sources.append(oi_yolo)
    print(f"Open Images added ({n} images)")
except Exception as e:
    print("[warn] Open Images skipped:", e)

In [ ]:
#@title 5) Merge into one YOLO dataset
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def load_names(root: Path):
    for fname in ("data.yaml", "dataset.yaml"):
        yml = root / fname
        if yml.is_file():
            data = yaml.safe_load(yml.read_text())
            names = data.get("names")
            if isinstance(names, dict):
                return {int(k): str(v) for k, v in names.items()}
            if isinstance(names, list):
                return {i: str(n) for i, n in enumerate(names)}
    return {}

def find_splits(root: Path):
    splits = {}
    for split in ("train", "val", "valid", "test"):
        a, b = root / "images" / split, root / "labels" / split
        if a.is_dir() and b.is_dir():
            splits["val" if split == "valid" else split] = (a, b)
        a, b = root / split / "images", root / split / "labels"
        if a.is_dir() and b.is_dir():
            splits["val" if split == "valid" else split] = (a, b)
    for split in ("train", "valid", "test"):
        folder = root / split
        if folder.is_dir() and any(p.suffix.lower() in IMG_EXTS for p in folder.iterdir()):
            key = "val" if split == "valid" else "train"
            splits.setdefault(key, (folder, folder))
    return splits

if MERGED.exists():
    shutil.rmtree(MERGED)
for split in ("train", "val"):
    (MERGED / "images" / split).mkdir(parents=True)
    (MERGED / "labels" / split).mkdir(parents=True)

box_counter = Counter()
image_idx = 0
random.seed(42)

for src in sources:
    root = src
    if not (src / "data.yaml").exists():
        for k in src.iterdir():
            if k.is_dir() and (k / "data.yaml").exists():
                root = k
                break
    names = load_names(root)
    if not names and root.name == "openimages_yolo":
        names = {i: n for i, n in enumerate(CLASS_NAMES)}
    if not names:
        print("[skip no names]", root)
        continue
    splits = find_splits(root)
    if not splits:
        print("[skip no splits]", root)
        continue
    print("[merge]", root.name, list(splits))

    items = []
    for split, (img_dir, lbl_dir) in splits.items():
        for img in sorted(p for p in img_dir.rglob("*") if p.suffix.lower() in IMG_EXTS):
            items.append(("train" if split == "test" else split, img, lbl_dir / f"{img.stem}.txt"))

    if "val" not in {s for s, _, _ in items}:
        random.shuffle(items)
        cut = max(1, int(len(items) * 0.15))
        items = [("val" if i < cut else "train", im, lb) for i, (_, im, lb) in enumerate(items)]

    for split, img, lbl in items:
        if not lbl.is_file():
            continue
        kept = []
        for line in lbl.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            old_id = int(float(parts[0]))
            if root.name == "openimages_yolo":
                if 0 <= old_id < len(CLASS_NAMES):
                    kept.append(f"{old_id} " + " ".join(parts[1:5]))
                    box_counter[CLASS_NAMES[old_id]] += 1
                continue
            cname = canonical(names.get(old_id, str(old_id)))
            if cname is None:
                continue
            cid = CLASS_TO_ID[cname]
            kept.append(f"{cid} " + " ".join(parts[1:5]))
            box_counter[cname] += 1
        if not kept:
            continue
        stem = f"{image_idx:06d}"
        shutil.copy2(img, MERGED / "images" / split / f"{stem}{img.suffix.lower()}")
        (MERGED / "labels" / split / f"{stem}.txt").write_text("\n".join(kept) + "\n")
        image_idx += 1

(MERGED / "data.yaml").write_text(yaml.safe_dump({
    "path": str(MERGED),
    "train": "images/train",
    "val": "images/val",
    "names": {i: n for i, n in enumerate(CLASS_NAMES)},
}, sort_keys=False))

print("Images:", image_idx)
print("Boxes:", dict(box_counter))
assert image_idx > 50, "Too few images — fix downloads and re-run"

In [ ]:
#@title 6) Train YOLOv8n — low-memory (avoids Colab crashes)
# If Colab still crashes: lower BATCH to 4, or IMGSZ to 320.

import gc
import torch
from pathlib import Path
from ultralytics import YOLO

WORK = Path("/content/raasta")
MERGED = WORK / "merged"
assert (MERGED / "data.yaml").exists(), "Run merge cell first"

# Free RAM/GPU from earlier download/merge cells
gc.collect()
torch.cuda.empty_cache()

MODEL = "yolov8n.pt"
EPOCHS = 60          # enough for FYP v1; raise later if stable
IMGSZ = 416          # 640 often OOMs on free Colab
BATCH = 8            # drop to 4 if it still crashes
WORKERS = 2

model = YOLO(MODEL)
model.train(
    data=str(MERGED / "data.yaml"),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    workers=WORKERS,
    device=0,
    project=str(WORK / "runs"),
    name="raasta_m2_m5",
    exist_ok=True,
    patience=15,
    cache=False,       # cache=True can crash Colab RAM
    amp=True,
    plots=False,       # less RAM during train
    save_period=10,    # resume-friendly checkpoints
    mosaic=0.5,        # lighter than 1.0
    close_mosaic=10,
    fliplr=0.5,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
)

BEST = WORK / "runs" / "raasta_m2_m5" / "weights" / "best.pt"
print("Best:", BEST)
metrics = YOLO(str(BEST)).val(
    data=str(MERGED / "data.yaml"),
    imgsz=IMGSZ,
    batch=BATCH,
    workers=WORKERS,
    plots=False,
)
print("mAP50:", float(metrics.box.map50))
print("mAP50-95:", float(metrics.box.map))

In [ ]:
#@title 7) Export TFLite + prepare downloads
best = YOLO(str(BEST))
export_path = best.export(format="tflite", imgsz=320)

shutil.copy2(BEST, "/content/raasta_m2_m5.pt")
shutil.copy2(export_path, "/content/raasta_m2_m5.tflite")

print("Download these from the left Files panel:")
print("  /content/raasta_m2_m5.pt")
print("  /content/raasta_m2_m5.tflite")
print()
print("Copy TFLite to:")
print(r"  E:\Android\projects\raasta_app\assets\models\raasta_m2_m5.tflite")
print()
print("Then tell the agent: TFLite is ready — wire detector into Flutter")

### Done

1. Write down **mAP50** from cell 6  
2. Download `raasta_m2_m5.tflite`  
3. Put it in `assets/models/`  
4. Come back to Cursor and say **TFLite is ready**

**Accuracy tip:** after this public-data train, label ~150 phone frames from Islamabad/Rawalpindi in Roboflow and fine-tune 20 more epochs — biggest jump for your viva demo.